In [7]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import altair as alt
alt.data_transformers.disable_max_rows()
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import json
import plotly.express as px
from sklearn.preprocessing import LabelEncoder

# { npm components } -------------------------------------------------------------------------------------------------------------------- #
import dtl
import selector
import bar_plt
import widgets
import radar
# { npm components } -------------------------------------------------------------------------------------------------------------------- #

anime = pd.read_csv("../data/preprocessed_anime.csv")
# sampling
anime = anime.sample(n=3000, random_state=42)

# { Consts } ---------------------------------------------------------------------------------------------------------------------------- #
root_container_style = {
    'position': 'absolute',
    'top': '0',
    'left': '0',
    'right': '0',
    'bottom': '0',
    'background-color': '#ECECEC',
    'overflow': 'hidden',
}
filter_container_style = {
    'position': 'relative',
    'width': '100%',
    'maxWidth': '256px',
    'marginBottom': '1rem',
    'height': 'auto',
} 
genre_filter_container_style = {
    'position': 'relative',
    'width': '100%',
    'maxWidth': '256px',
    'marginBottom': '1rem',
    'height': 'auto',
}

# list of string
type_options = ['All'] + [anime_type for anime_type in anime['Type'].dropna().unique()]
genre_options = ['All'] + sorted(set([g.strip() for genres in anime['Genres'].dropna() for g in genres.split(',')]))
studio_options = ['All'] + sorted(set([s.strip() for studios in anime['Studios'].dropna() for s in studios.split(',')]))
# { Consts } ---------------------------------------------------------------------------------------------------------------------------- #

# { Data preprocessing } ---------------------------------------------------------------------------------------------------------------- # 
# anime['start_date'] = pd.to_datetime(anime['start_date'])
# anime['end_date'] = pd.to_datetime(anime['end_date'])
def data_preprocessing(df, selected_filters=None):
    df = df.copy()
    df['start_date'] = pd.to_datetime(df['start_date'])
    df['end_date'] = pd.to_datetime(df['end_date'])
    
    if selected_filters[0] and selected_filters[0] != 'All':
        df = df[df['Type'] == selected_filters[0]]
            
    if selected_filters[2]:
        if selected_filters[2] != 'All':
            df = df[df['Studios'] == selected_filters[2]]
    
    # make sure the explode is done after type and studio filter
    df = df.assign(Genres=df['Genres'].str.split(', ')).explode('Genres')

    # make sure the genre filter happen after the explode
    if selected_filters[1]:
        if selected_filters[1] != 'All':
            df = df[df['Genres'] == selected_filters[1]]    

    return df
# { Data preprocessing } ---------------------------------------------------------------------------------------------------------------- # 

# { Graph generation functions } ======================================================================================================== #
def generate_heatmap(df):
    """
    Create a correlation heatmap based on filtered data
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The anime dataset
    selected_type : str, optional
        The type of anime to filter by (e.g., 'TV', 'Movie', etc.)
    """
    # copy the dataframe
    df = df.copy()
    
    # Explode Genres with comma (assuming multi-genre entries)
    df = df.assign(Genres=df['Genres'].str.split(', ')).explode('Genres')
    
    # Convert Aired to datetime and extract year
    df['Aired'] = pd.to_datetime(df['Aired'], errors='coerce')
    df['Aired'] = df['Aired'].dt.year
    
    # Define important columns
    important_columns = ['Genres', 'Type', 'Episodes', 'Aired', 'Studios', 'Members', 'Score', 'Popularity']
    df_subset = df[important_columns].dropna()
    
    # Convert categorical columns to integers using LabelEncoder
    le = LabelEncoder()
    df_subset['Genres'] = le.fit_transform(df_subset['Genres'])
    df_subset['Type'] = le.fit_transform(df_subset['Type'])
    df_subset['Studios'] = le.fit_transform(df_subset['Studios'])

    # negate the popularity for making it more intuitive for ranking
    df_subset['Popularity'] = -df_subset['Popularity']
    
    # Define target variables
    target_vars = ['Score', 'Popularity']
    
    # Get all columns for correlation
    all_columns = df_subset.columns
    
    # Calculate correlation matrix
    correlation_matrix = df_subset[all_columns].corr()
    
    # Filter out the row with column and column with target variables   
    corr_with_targets = correlation_matrix.loc[all_columns, target_vars]
    
    # Heatmap creation
    fig = go.Figure(data=go.Heatmap(
        z=corr_with_targets.values,
        x=target_vars,
        y=all_columns,
        hoverongaps=False,
        zmin=-1, zmax=1,
        colorscale = 'Blues',
        text=np.round(corr_with_targets.values, 2),
        texttemplate='%{text}',
        textfont={"size": 12},
        showscale=False,
    ))

    fig.update_layout(
        title="Correlation Heatmap For Finding Impactful Predictors",
        xaxis_title="Target Variables",
        yaxis_title="Predictors",
        height=500, 
        width=500,
        xaxis=dict(
            showgrid=False,
            zeroline=False,
            # show the predictors names as labels
            showticklabels=True
        ),
        yaxis=dict(
            showgrid=False,
            zeroline=False,
            showticklabels=True 
        ),
        margin=dict(l=50, r=50, t=50, b=50),
        # Transparent background
        paper_bgcolor='rgba(0,0,0,0)',  
        plot_bgcolor='rgba(0,0,0,0)', 
    )

    return fig
def generate_radar(df):
    df_exploded = df.assign(Genres=df['Genres'].str.split(', ')).explode('Genres')

    # Count the frequency of each genre and select the top 10 most frequent ones
    top_genres = df_exploded['Genres'].value_counts().head(10).index

    # Filter dataset to include only the top 10 genres
    df_exploded = df_exploded[df_exploded['Genres'].isin(top_genres)]

    # Select relevant numerical columns
    columns = ['Score', 'Members', 'Popularity', 'Completed', 'On-Hold', 'Dropped']

    # Aggregate by genre, computing the mean for each variable
    df_genre_avg_original = df_exploded.groupby('Genres')[columns].mean().reset_index()

    # Store the original mean data before normalization
    df_genre_avg = df_genre_avg_original.copy()

    # Normalize values for better visualization (Min-Max Scaling)
    df_genre_avg[columns] = (df_genre_avg[columns] - df_genre_avg[columns].min()) / \
                            (df_genre_avg[columns].max() - df_genre_avg[columns].min())

    json_data = [
        {
            "genre": row["Genres"],
            "Score": round(row["Score"], 3),
            "Members": round(row["Members"], 3),
            "Popularity": round(row["Popularity"], 3),
            "Completed": round(row["Completed"], 3),
            "OnHold": round(row["On-Hold"], 3),
            "Dropped": round(row["Dropped"], 3),
        }
        for _, row in df_genre_avg.iterrows()
    ]

    return radar.Radar(
        id='dash-radar-chart',
        data=json_data,
    )
def generate_bar(df):
    # Create and process data
    genre_avg_score = df.groupby('Genres')['Score'].mean().round(2).reset_index()
    # sort by score descending
    genre_avg_score = genre_avg_score.sort_values(by='Score', ascending=False)
    
    # Take top 10 genres (keeping it consistent with your original)
    top_genres = 10
    bar_height = 30
    genre_avg_score = genre_avg_score.head(top_genres)
    
    res = [{'genre': row['Genres'], 'score': row['Score']} for index, row in genre_avg_score.iterrows()]
    
    return bar_plt.BarPlt(
        id='dash-bar-chart',
        data=res,
    )
def generate_timeline_component(df):
    def generate_anime_count_by_date(df):
        all_dates = []
        
        df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
        df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')
        
        df = df.dropna(subset=['start_date', 'end_date'])
        
        for _, row in df.iterrows():
            date_range = pd.date_range(row['start_date'], row['end_date'], freq='D')
            all_dates.extend(date_range)
            
        date_counts = pd.DataFrame(all_dates, columns=['date'])
        date_counts = date_counts.groupby('date').size().reset_index(name='anime_count')
        date_counts = date_counts.iloc[::30, :]

        date_counts = date_counts.to_numpy()
        date_counts = np.array(date_counts)
        date_counts = date_counts.tolist()
        
        x = []
        y = []

        for i in date_counts:
            x.append(str(i[0])[:10])
            y.append(i[1])


        return x[::3], y[::3]
    def generate_average_score_by_date(df):
        df = df.iloc[::60, :]
        df['Score'] = pd.to_numeric(df['Score'], errors='coerce')
        
        all_dates_scores = []
        for _, row in df.iterrows():
            date_range = pd.date_range(row['start_date'], row['end_date'], freq='D')
            all_dates_scores.extend([(date, row['Score']) for date in date_range])

        # Convert to DataFrame and calculate average score per date
        date_scores_df = pd.DataFrame(all_dates_scores, columns=['date', 'score'])
        average_scores = date_scores_df.groupby('date').agg(
            avg_score=('score', 'mean'),
            anime_count=('score', 'count')
        ).reset_index()

        average_scores
        #break into x and y x is date y is avg_score both are arrays
        # convert date to string only date part
        x = average_scores['date'].tolist()
        x = [str(i).split
            (' ')[0] for i in x]
        y = average_scores['avg_score'].tolist()
        
        return x[::7], y[::7]
    
    count_x, count_y = generate_anime_count_by_date(df)
    score_x, score_y = generate_average_score_by_date(df)
    
    combined_x = sorted(set(count_x + score_x))
    count_dict = dict(zip(count_x, count_y))
    score_dict = dict(zip(score_x, score_y))
    aligned_count_y = [count_dict.get(date, 0) for date in combined_x]
    aligned_score_y = [score_dict.get(date, 0) for date in combined_x]
    
    return dtl.Dtl(
        id='dash-timeline',
        countX=combined_x,
        countY=aligned_count_y,
        scoreX=combined_x,
        scoreY=aligned_score_y
    )

def generate_widgets(df):
    def get_top_3_animes(df):
        top_scores = (
            df.dropna(subset=['Name', 'Score'])
            .sort_values(by='Score', ascending=False)
            .drop_duplicates(subset='Name')
            .head(3)[['Name', 'Score']])
    
        return [{"title": row["Name"], "score": f"{row['Score']:.2f}"} for _, row in top_scores.iterrows()]

    def get_average_score(df):
        return round(df["Score"].mean(), 2)
    
    top_3_animes = get_top_3_animes(df)
    average_score = get_average_score(df)
    
    return widgets.Widgets(
        id='dash-widgets',
        top_3=top_3_animes,
        average_score=average_score
    )
# { Graph generation functions } ======================================================================================================== #

# { Dash App } -------------------------------------------------------------------------------------------------------------------------- #
# initialize dash app
app = dash.Dash(__name__)

app.layout = html.Div([
    # Main content container
    html.Div([
        # Top row with radar and bar charts
        html.Div([      
            dcc.Graph(
                id='heatmap-graph',
                style={
                    'width': '30%',
                    'height': '400px',
                }
            ),
        ], style={
            'display': 'flex',
            'justifyContent': 'space-between',
            'width': '100%',
            'marginBottom': '0.5rem',
        }),
    ], style={
        'position': 'absolute',
        'top': '0',
        'left': '0',
        'right': '0',
        'bottom': '0',
    }),
    html.Div(id='dash-radar-chart-container'),
    html.Div(id='dash-timeline-container'),
    html.Div(id='dash-bar-chart-container'),
    html.Div(id='dash-widgets-container'),
    selector.Selector(
        id='selector',
        values=['All', 'All', 'All'],
        typeOptions=type_options,
        genreOptions=genre_options,
        studioOptions=studio_options    
    ),
], style=root_container_style)

@app.callback(
    [Output('heatmap-graph', 'figure'),
     Output('dash-radar-chart-container', 'children'),
     Output('dash-bar-chart-container', 'children'),
     Output('dash-timeline-container', 'children'),
     Output('dash-widgets-container', 'children')],
    [Input('selector', 'values')]
)
def update_graphs(selected_filters):
    processed_anime = data_preprocessing(anime, selected_filters)
    heatmap_graph = generate_heatmap(processed_anime)
    radar_graph = generate_radar(processed_anime)
    bar_chart = generate_bar(processed_anime)
    time_line_graph = generate_timeline_component(processed_anime)
    widgets = generate_widgets(processed_anime)
    return heatmap_graph, radar_graph, bar_chart, time_line_graph, widgets

if __name__ == '__main__':
    app.run_server(debug=False)
# { Dash App } -------------------------------------------------------------------------------------------------------------------------- #


C:\Users\zheng\AppData\Local\Temp\ipykernel_97588\303485460.py:255: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

